# AstroVis

**AstroVis** is a modular astrophysical visualization framework designed to transform simulation data into physically interpretable renders in Blender.


## Architecture Overview

AstroVis consists of four layers:

### 1. Backend

Blender-independent scientific data manipulation.

- Store particle and grid data  
- Convert SPH particle data → grid  
- Convert grid → surface or volume  
- Runs inside or outside Blender  

### 2. Blender_Import

Import processed data into Blender.

- Load particle animation  
- Load surface mesh animation  
- Load VDB volume animation  

### 3. Blender_Effect

Rendering control inside Blender.

- Geometry Nodes  
- Shading  
- Scene & lighting  
- Object management  

### 4. Blender_Export

Export final output.

- Video rendering  
- Sketchfab export  

--- 
## 1. Backend 

The **Backend layer** performs all physics-aware data processing and can be executed entirely outside Blender.

### Import Backend

If running outside Blender, use:

```python
from AstroVis.backend import *
```

This avoids importing Blender-specific modules.

---
### 1.1: Loading SPH Particle Data

The `load_particles()` function automatically extracts:

- Coordinates  
- Smoothing lengths  
- Masses  
- Densities  

You only need to specify:

- `ptype`
- Additional fields (optional)


#### Basic Example 

In [ ]:
import yt

ds = yt.load("output_7bin_HHeZdust_real_0060.hdf5")
stromgren = load_particles(ds, ptype="PartType0", fields=["Mass"])
print(stromgren)


: 

#### Discover Available Fields


In [10]:
print(ds.field_list)
print(ds.derived_field_list)

[('PartType0', 'AtomicHydrogenMasses'), ('PartType0', 'AveragedStarFormationRates'), ('PartType0', 'ComptonYParameters'), ('PartType0', 'Coordinates'), ('PartType0', 'Densities'), ('PartType0', 'DensitiesAtLastAGNEvent'), ('PartType0', 'DensitiesAtLastSupernovaEvent'), ('PartType0', 'DiffusionParameters'), ('PartType0', 'ElectronNumberDensities'), ('PartType0', 'ElementMassFractions'), ('PartType0', 'EnergiesReceivedFromAGNFeedback'), ('PartType0', 'Entropies'), ('PartType0', 'HIIregionsEndTime'), ('PartType0', 'HIIregionsStarIDs'), ('PartType0', 'InternalEnergies'), ('PartType0', 'IronMassFractionsFromSNIa'), ('PartType0', 'IsotropicPhotonDensity'), ('PartType0', 'LaplacianInternalEnergies'), ('PartType0', 'LastAGNFeedbackTimes'), ('PartType0', 'LastFOFHaloMasses'), ('PartType0', 'LastFOFHaloMassesTimes'), ('PartType0', 'LastISMTimes'), ('PartType0', 'LastKineticEarlyFeedbackTimes'), ('PartType0', 'LastSNIIKineticFeedbackTimes'), ('PartType0', 'LastSNIIKineticFeedbackvkick'), ('PartTy

#### Using Callable Fields

Custom fields may be defined dynamically.

In [11]:
import yt
import numpy as np

ds = yt.load("output_7bin_HHeZdust_real_0060.hdf5")
    
ad = ds.all_data()

def HI_fraction():
    return np.ascontiguousarray(ad['PartType0', "SpeciesFractions"][:, 1])
def HII_fraction():
    return np.ascontiguousarray(ad['PartType0', "SpeciesFractions"][:, 2])

stromgren = load_particles(ds, ptype="PartType0", fields={"HI_fraction": HI_fraction, "HII_fraction": HII_fraction})

yt : [INFO     ] 2026-03-05 03:21:53,842 Assuming length units are in physical centimetres
yt : [INFO     ] 2026-03-05 03:21:53,872 Parameters: current_time              = 0.00015374999999999983
yt : [INFO     ] 2026-03-05 03:21:53,873 Parameters: domain_dimensions         = None
yt : [INFO     ] 2026-03-05 03:21:53,873 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-05 03:21:53,874 Parameters: domain_right_edge         = [0.044 0.044 0.044]
yt : [INFO     ] 2026-03-05 03:21:53,874 Parameters: cosmological_simulation   = 0
yt : [INFO     ] 2026-03-05 03:21:53,897 Allocating for 2.621e+05 particles


#### Conversion

The following data can be imported into blender through the Script
1. Particle Mesh
2. Surface Mesh
3. VDB Volume

This script allow conversion from sph_particle_to_grid, grid_to_surface and grid_to_vdb

In [ ]:
## From sph_particle_to_grids

stromgren_grid = sph_to_grid(stromgren, field = "HI_fraction", res = 128)
print(stromgren_grid)

## From grid to surface
stromgen_surface = grid_to_surface(stromgren_grid, threshold = 0.5, field = "HI_fraction")

## Export the surface data
np.savez("stromgren_surface.npz", stromgen_surface)


GridBlock(block_id=0, left_edge=(0.0, 0.0, 0.0), right_edge=(1.0, 1.0, 1.0), dims=(128, 128, 128), fields={'HI_fraction': array([[[0.9989122 , 0.99891245, 0.9989131 , ..., 0.99899167,
         0.99898237, 0.9989757 ],
        [0.99891895, 0.9989219 , 0.99892473, ..., 0.99901533,
         0.9990116 , 0.9990113 ],
        [0.99898046, 0.9989826 , 0.9989707 , ..., 0.9990308 ,
         0.9990318 , 0.99903655],
        ...,
        [0.9987756 , 0.99878323, 0.9987933 , ..., 0.9987741 ,
         0.9987679 , 0.9987644 ],
        [0.9987783 , 0.9987868 , 0.9987976 , ..., 0.9987805 ,
         0.9987712 , 0.99876493],
        [0.99878085, 0.99879074, 0.99880314, ..., 0.9987876 ,
         0.9987755 , 0.9987662 ]],

       [[0.9989184 , 0.99891365, 0.998911  , ..., 0.9989829 ,
         0.99897563, 0.9989707 ],
        [0.99893105, 0.9989262 , 0.9989234 , ..., 0.9990132 ,
         0.9990073 , 0.9990052 ],
        [0.9989727 , 0.9989711 , 0.9989646 , ..., 0.9990382 ,
         0.9990348 , 0.9990348 ],

Then import in Blender:

```python
stromgren_surface = np.load("stromgren_surface.npz", allow_pickle=True)
setup_surface_mesh(stromgren_surface, "HI_front", position_scale=20)
```
---

### Running within python console of Blender 
If the file size are not too big, it will be more convenient if calling the function above through blender directly.

```python
import AstroVis
import yt
import numpy as np

ds = yt.load(os.path.join(os.path.dirname(bpy.data.filepath), "output_7bin_HHeZdust_real_0060.hdf5"))
data = ds.all_data()
def HI_fraction():
    return np.ascontiguousarray(data['PartType0', "SpeciesFractions"][:, 1])

star = load_particles(ds, ptype = 'PartType0', fields = {"HI_fraction": HI_fraction})
HI_grid = sph_to_grid(star, field = "HI_fraction")
surface_data = grid_to_surface(HI_grid, 0.5)
obj = setup_surface_mesh(surface_data, "trial", position_scale=20)
```
---

### 1.2: Loading Volume Data
The data format storing particle data and volume data is different.

#### Basic Example

In [13]:
import yt

ds = yt.load("DiskForm.hydro.00140.athdf")

protostar = load_volume(ds, vtype="gas", fields=["density"])
print(protostar.levels[0].blocks[0].fields['density'])

yt : [WARNING  ] 2026-03-05 03:22:00,141 Assuming 1.0 = 1.0 cm
yt : [WARNING  ] 2026-03-05 03:22:00,142 Assuming 1.0 = 1.0 s
yt : [WARNING  ] 2026-03-05 03:22:00,145 Assuming 1.0 = 1.0 g


yt : [WARNING  ] 2026-03-05 03:22:00,146 Assuming 1.0 = 1.0 K
yt : [INFO     ] 2026-03-05 03:22:00,190 Parameters: current_time              = 7.000000462167005
yt : [INFO     ] 2026-03-05 03:22:00,191 Parameters: domain_dimensions         = [128 128 128]
yt : [INFO     ] 2026-03-05 03:22:00,191 Parameters: domain_left_edge          = [-0.5 -0.5 -0.5]
yt : [INFO     ] 2026-03-05 03:22:00,192 Parameters: domain_right_edge         = [0.5 0.5 0.5]
yt : [INFO     ] 2026-03-05 03:22:00,192 Parameters: cosmological_simulation   = 0


[[[0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  [0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  [0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  ...
  [0.05314083 0.05314083 0.05314083 ... 0.03321644 0.0328032  0.0324584 ]
  [0.05377407 0.05377407 0.05377407 ... 0.03351157 0.03305053 0.03267078]
  [0.05435328 0.05435328 0.05435328 ... 0.03373671 0.03323432 0.03282527]]

 [[0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  [0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  [0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  ...
  [0.05314083 0.05314083 0.05314083 ... 0.03321644 0.0328032  0.0324584 ]
  [0.05377407 0.05377407 0.05377407 ... 0.03351157 0.03305053 0.03267078]
  [0.05435328 0.05435328 0.05435328 ... 0.03373671 0.03323432 0.03282527]]

 [[0.05255964 0.05255964 0.05255964 ... 0.05711126 0.05745434 0.05765425]
  [0.05255964 0.052559

#### Grid → VDB Conversion

To import a volumetric object in blender, we need to produce a .vdb file.

Requirements for the conversion:

- Python 3.8  

```python
convert_vdb(protostar.levels[0].blocks[0].fields["density"])
```

---

## 2. Import into Blender

After the dataset has been prepared using the **Backend**, the processed data can be imported into Blender.

AstroVis provides three primary import functions for loading different types of data:

- **Particle meshes**
- **Surface meshes**
- **Volume data**

⚠ These functions must be executed **inside Blender**, since they rely on the `bpy` module.


### Basic Examples

Import **SPH particle data** as a particle mesh:

```python
setup_particle_mesh(stromgren, "HI_fraction", position_scale=20)
```

Import a **surface mesh** generated from grid data:

```python
setup_surface_mesh(stromgren_surface, "HI_0.5_fraction", position_scale=20)
```

**Parameters**

- `stromgren` / `stromgren_surface` — data object produced by the Backend 
- `"HI_fraction"` — name of the object in Bldender  
- `position_scale` — scaling factor applied to particle coordinates

---

### Animation Support

AstroVis automatically generates **frame-by-frame animations** when the imported dataset is provided as a **list of frame data**.

Each element in the list represents one frame of the animation.


#### Step 1 — Prepare Surface Data for Each Frame

```python
stromgren_surface_frame_data = []

for file in file_paths:

    ds = yt.load(file)
    ad = ds.all_data()

    def HI_fraction():
        return np.ascontiguousarray(ad['PartType0', "SpeciesFractions"][:, 1])

    stromgren = load_particles(
        ds,
        ptype="PartType0",
        fields={
            "HI_fraction": HI_fraction
        }
    )

    HI_grid = sph_to_grid(stromgren, field="HI_fraction")

    stromgren_surface = grid_to_surface(HI_grid, 0.5)

    stromgren_surface_frame_data.append(stromgren_surface)
```

This loop processes a sequence of simulation outputs and converts them into **surface meshes for each frame**.


#### Step 2 — Import Animation into Blender

```python
setup_surface_mesh(
    stromgren_surface_frame_data,
    "HI_0.5_fraction",
    position_scale=20
)
```
---
### Higher-Level API

For complex workflows involving many files or datasets, AstroVis also provides **higher-level helper functions** that automate file discovery, loading, and import into Blender.

---

## 3. Blender Effects

The **Blender_Effect** module controls how imported objects are rendered inside Blender.  
It includes utilities for:

- Geometry node generation
- Shader creation
- Scene configuration

These tools operate on objects that have already been imported into Blender.

---

### 3.1 Geometry Nodes

The following functions generate predefined **Geometry Node networks** for common visualization tasks.

These node setups convert particle data into renderable structures such as **volumes or meshes**.

⚠ Note:  
Geometry nodes only define the **geometry transformation**.  
To render the result properly, an appropriate **shader must be assigned** (see Section 3.2).

#### SPH Particle → Volume

Creates a volume representation from SPH particle data.

```python
sph_point_to_volume(
    obj: typing.Union[str, bpy.types.Object],
    attribute_name: str | None = None,
    material_name: str | None = None,
    voxel_size: float = 0.02,
    density: float = 0.2,
    math_multiplier: float = 0.03
)
```

**Parameters**

- `obj` – Blender object containing the particle dataset  
- `attribute_name` – attribute used to control volume density  
- `material_name` – name of the material applied to the generated volume  
- `voxel_size` – resolution of the generated volume grid  
- `density` – base density scaling  
- `math_multiplier` – additional multiplier applied in the node graph  

---

#### SPH Particle → Mesh Surface

Generates a mesh surface from particle data.

```python
sph_point_to_mesh(
    obj,
    radius_attr: str = None,
    material_name: str = None,
    voxel_size: float = 0.3,
    density: float = 1.0,
    radius_multiplier: float = 1.1
)
```

**Parameters**

- `obj` – particle object to be converted  
- `radius_attr` – particle attribute used to define influence radius  
- `material_name` – material assigned to the generated mesh  
- `voxel_size` – grid resolution used for surface reconstruction  
- `density` – density threshold for surface extraction  
- `radius_multiplier` – scaling applied to particle radii  

---

#### Attribute / Proximity Selection

This node setup creates a **filtered object** based on attribute conditions or spatial proximity.

```python
select(
    obj,
    new_object_name: str,
    mode: str = "attribute",  # "attribute" or "proximity"
    attribute_name: str = None,
    compare_value: float = None,
    operation: str = "GREATER",  # GREATER, LESS, EQUAL, NOT_EQUAL
    target_obj=None,
    target_element: str = "POINTS",  # POINTS, EDGES, FACES
    distance: float = 1.0,
)
```

**Modes**

- `attribute` – select elements based on attribute values  
- `proximity` – select elements based on distance to another object  

**Key Parameters**

- `new_object_name` – name of the resulting filtered object  
- `attribute_name` – attribute used for comparison  
- `compare_value` – threshold value for selection  
- `operation` – comparison operator  
- `target_obj` – reference object used in proximity mode  
- `distance` – selection radius  

---

### 3.2 Shading

AstroVis provides utilities to automatically generate **physically interpretable shaders** for visualization.

Two types of shaders are supported:

- **Volume shaders** (for VDB / volumetric rendering)
- **Mesh shaders** (for surfaces)

Multiple shaders can be generated simultaneously.  


#### Volume Shaders

Creates emission-based shaders for volumetric visualization.

```python
create_volume_shaders(
    species_names: typing.List[str],
    species_colors: typing.Tuple = None,
    species_alpha: float = 1.0,
    emission_multiplier: float = 2.0,
    density_attribute: str = "density"
) -> typing.Dict[str, bpy.types.Material]
```

**Parameters**

- `species_names` – list of species to generate materials for  
- `species_colors` – optional custom color assignment  
- `species_alpha` – transparency factor  
- `emission_multiplier` – emission intensity scaling  
- `density_attribute` – attribute used as the density source  

Returns a dictionary mapping:

```
species_name → Blender Material
```

#### Mesh Shaders

Creates surface shaders for mesh-based visualizations.

```python
create_mesh_shaders(
    species_names: typing.List[str],
    base_color: typing.Tuple = (0.54, 0.20, 0.0, 1.0),
    emission_color: typing.Tuple = (0.57, 0.15, 0.0, 1.0),
    emission_strength: float = 0.15,
) -> typing.Dict[str, bpy.types.Material]
```

**Parameters**

- `species_names` – list of mesh materials to generate  
- `base_color` – diffuse color of the surface  
- `emission_color` – emission color used for glow effects  
- `emission_strength` – emission intensity  

Returns a dictionary mapping:

```
species_name → Blender Material
```